In [1]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.1 MB/s eta 0:00:00


In [2]:
%%writefile /kaggle/working/recproj2_candidates.py

""""Кандидаты из истории сессии, Item2Vec и популярных товаров."""

from collections import defaultdict

import faiss
import numpy as np
from gensim.models import Word2Vec


def load_model_and_build_index(model_path, M=16, ef_construction=200, ef_search=128):
    """Загружает Item2Vec из файла и строит индекс похожих товаров."""
    model = Word2Vec.load(str(model_path))
    if hasattr(model, 'syn1neg'):
        del model.syn1neg

    vectors = np.array(model.wv.vectors, dtype=np.float32, copy=True)
    faiss.normalize_L2(vectors)
    index = faiss.IndexHNSWFlat(
        vectors.shape[1], M, faiss.METRIC_INNER_PRODUCT
    )
    index.hnsw.efConstruction = ef_construction
    index.add(vectors)
    index.hnsw.efSearch = ef_search
    del vectors

    return model, index, model.wv.index_to_key


def generate_candidates(session_aids, session_types, session_ts, event_type,
                        model, index, idx_to_aid, *, popular_items,
                        type_weights, top_k=150, last_n=10,
                        return_metadata=False):
    """Возвращает до top_k товаров в порядке убывания оценки.

    session_aids, session_types и session_ts описывают известную историю
    одной сессии в порядке событий. Идентификаторы товаров — строки.
    popular_items содержит списки популярных товаров по типам событий,
    type_weights — веса действий в истории. Модель и индекс создаются
    заранее и переиспользуются для всех сессий.

    При return_metadata=True возвращает (candidates, metadata_by_aid).
    Для каждого кандидата сохраняются оценка истории, исходное сходство
    Item2Vec и флаги источников. Популярный источник учитывается, когда
    его список используется для дополнения кандидатов или пустой истории.
    Порядок кандидатов одинаков при обоих вариантах возврата.
    """
    popular = popular_items.get(event_type, [])
    if not session_aids:
        candidates = popular[:top_k]
        if not return_metadata:
            return candidates
        metadata = {
            aid: {
                'history_score': 0.0,
                'item2vec_retrieval_score': 0.0,
                'from_history': 0,
                'from_item2vec': 0,
                'from_popular': 1,
                'candidate_source_count': 1,
            }
            for aid in candidates
        }
        return candidates, metadata

    scores = defaultdict(float)
    min_ts = min(session_ts)
    max_ts = max(session_ts)
    time_span = max(max_ts - min_ts, 1.0)

    for aid, action, ts in zip(session_aids, session_types, session_ts):
        time_weight = (ts - min_ts) / time_span
        scores[aid] += type_weights.get(action, 1.0) * time_weight

    history_scores = dict(scores) if return_metadata else {}
    retrieval_scores = {}
    popular_candidates = set()

    recent_aids = session_aids[-last_n:]
    recent_types = session_types[-last_n:]
    recent_ts = session_ts[-last_n:]
    vectors = []
    weights = []

    for aid, action, ts in zip(recent_aids, recent_types, recent_ts):
        if aid in model.wv.key_to_index:
            time_weight = (ts - min_ts) / time_span
            vectors.append(model.wv[aid])
            weights.append(type_weights.get(action, 1.0) * time_weight)

    if vectors:
        weights = np.asarray(weights, dtype=np.float32)
        weight_sum = weights.sum()
        if weight_sum > 0:
            weights /= weight_sum
        else:
            weights[:] = 1.0 / len(weights)

        session_vector = np.average(vectors, axis=0, weights=weights)
        session_vector = np.asarray(session_vector, dtype=np.float32).reshape(1, -1)
        faiss.normalize_L2(session_vector)
        distances, indices = index.search(
            session_vector, min(2 * top_k, index.ntotal)
        )

        for item_index, similarity in zip(indices[0], distances[0]):
            if item_index == -1:
                continue
            aid = idx_to_aid[item_index]
            scores[aid] += max(float(similarity), 0.0)
            if return_metadata:
                retrieval_scores[aid] = float(similarity)

    if len(scores) < top_k:
        for aid in popular[:min(50, top_k)]:
            if return_metadata:
                popular_candidates.add(aid)
            if aid not in scores:
                scores[aid] += 0.5
            if len(scores) >= top_k:
                break

    candidates = [
        aid for aid, _ in sorted(
            scores.items(), key=lambda pair: (-pair[1], pair[0])
        )[:top_k]
    ]
    if not return_metadata:
        return candidates

    metadata = {}
    for aid in candidates:
        from_history = int(aid in history_scores)
        from_item2vec = int(aid in retrieval_scores)
        from_popular = int(aid in popular_candidates)
        metadata[aid] = {
            'history_score': history_scores.get(aid, 0.0),
            'item2vec_retrieval_score': retrieval_scores.get(aid, 0.0),
            'from_history': from_history,
            'from_item2vec': from_item2vec,
            'from_popular': from_popular,
            'candidate_source_count': from_history + from_item2vec + from_popular,
        }
    return candidates, metadata

Writing /kaggle/working/recproj2_candidates.py
